THIS CODE can be runned only if we have a dedicated cluster
we will use Use Serverless + Lakeflow Declarative Pipeline in Continuous mode

In [0]:
# ============================================================
# RAILWAY PROJECT 3
# BRONZE -> SILVER CDC STREAMING
#
# Handles:
#   1. Initial full load
#   2. Incremental Bronze CDF processing
#   3. INSERT
#   4. UPDATE
#   5. DELETE
#   6. Deduplication
#   7. Streaming checkpoints
#
# Bronze:
#   motordatabricks.bronze.trains
#   motordatabricks.bronze.schedules
#   motordatabricks.bronze.maintenance_workorders
#
# Silver:
#   motordatabricks.silver.trains
#   motordatabricks.silver.schedules
#   motordatabricks.silver.maintenance_workorders
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "motordatabricks"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

CHECKPOINT_BASE = (
    "abfss://silver@railwaystoragehrk.dfs.core.windows.net/"
    "checkpoints/bronze_to_silver/"
)

TABLE_CONFIG = {

    "trains": {
        "bronze": f"{CATALOG}.{BRONZE_SCHEMA}.trains",
        "silver": f"{CATALOG}.{SILVER_SCHEMA}.trains",
        "key": "train_id",
        "columns": [
            "train_id",
            "train_number",
            "train_name",
            "train_type",
            "current_status",
            "current_station",
            "updated_at"
        ]
    },

    "schedules": {
        "bronze": f"{CATALOG}.{BRONZE_SCHEMA}.schedules",
        "silver": f"{CATALOG}.{SILVER_SCHEMA}.schedules",
        "key": "schedule_id",
        "columns": [
            "schedule_id",
            "train_id",
            "source_station",
            "destination_station",
            "departure_time",
            "arrival_time",
            "schedule_status",
            "updated_at"
        ]
    },

    "maintenance_workorders": {
        "bronze": f"{CATALOG}.{BRONZE_SCHEMA}.maintenance_workorders",
        "silver": f"{CATALOG}.{SILVER_SCHEMA}.maintenance_workorders",
        "key": "workorder_id",
        "columns": [
            "workorder_id",
            "train_id",
            "maintenance_type",
            "description",
            "priority",
            "status",
            "technician",
            "scheduled_date",
            "completed_date",
            "updated_at"
        ]
    }
}

In [0]:
# ============================================================
# COMMON TRANSFORMATION
# ============================================================

def clean_string_columns(df, columns):

    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    for col_name in string_columns:
        if col_name in columns:
            df = df.withColumn(
                col_name,
                F.when(
                    F.col(col_name).isNull(),
                    F.lit(None)
                ).otherwise(
                    F.trim(F.col(col_name))
                )
            )

    return df

In [0]:
# ============================================================
# DEDUPLICATE CDF EVENTS
# ============================================================

def prepare_cdf_batch(batch_df, key_column, columns):

    # --------------------------------------------------------
    # Remove update_preimage
    # We only need the final state of an UPDATE.
    # --------------------------------------------------------

    df = batch_df.filter(
        F.col("_change_type").isin(
            "insert",
            "update_postimage",
            "delete"
        )
    )

    # --------------------------------------------------------
    # Data quality:
    # Primary key must not be NULL
    # --------------------------------------------------------

    df = df.filter(
        F.col(key_column).isNotNull()
    )

    # --------------------------------------------------------
    # Clean string columns
    # --------------------------------------------------------

    df = clean_string_columns(df, columns)

    # --------------------------------------------------------
    # Event priority
    #
    # If multiple events for the same key exist in a batch,
    # the latest commit wins.
    #
    # DELETE gets highest priority if the same commit contains
    # multiple event types.
    # --------------------------------------------------------

    df = df.withColumn(
        "_event_priority",
        F.when(F.col("_change_type") == "delete", F.lit(3))
         .when(F.col("_change_type") == "update_postimage", F.lit(2))
         .when(F.col("_change_type") == "insert", F.lit(1))
         .otherwise(F.lit(0))
    )

    # --------------------------------------------------------
    # Keep latest event per primary key
    # --------------------------------------------------------

    window_spec = (
        Window
        .partitionBy(key_column)
        .orderBy(
            F.col("_commit_version").desc(),
            F.col("_event_priority").desc(),
            F.col("_commit_timestamp").desc()
        )
    )

    df = (
        df
        .withColumn("_row_number", F.row_number().over(window_spec))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number", "_event_priority")
    )

    return df

In [0]:
# ============================================================
# PROCESS ONE MICRO-BATCH
# ============================================================

def process_batch(batch_df, batch_id, table_name):

    config = TABLE_CONFIG[table_name]

    bronze_table = config["bronze"]
    silver_table = config["silver"]
    key_column = config["key"]
    columns = config["columns"]

    print("=" * 80)
    print(f"Processing table : {table_name}")
    print(f"Batch ID         : {batch_id}")
    print("=" * 80)

    # --------------------------------------------------------
    # Handle empty micro-batches
    # --------------------------------------------------------

    if batch_df.isEmpty():
        print(f"No changes for {table_name}")
        return

    # --------------------------------------------------------
    # Prepare CDF data
    # --------------------------------------------------------

    changes = prepare_cdf_batch(
        batch_df,
        key_column,
        columns
    )

    if changes.isEmpty():
        print(f"No valid changes for {table_name}")
        return

    # --------------------------------------------------------
    # Select only actual Silver columns + CDC metadata
    # --------------------------------------------------------

    changes = changes.select(
        *columns,
        "_change_type",
        "_commit_version",
        "_commit_timestamp"
    )

    # --------------------------------------------------------
    # Get target Delta table
    # --------------------------------------------------------

    target = DeltaTable.forName(
        spark,
        silver_table
    )

    # ========================================================
    # STEP 1 - DELETE
    # ========================================================

    deletes = (
        changes
        .filter(F.col("_change_type") == "delete")
        .select(key_column)
        .dropDuplicates([key_column])
    )

    if not deletes.isEmpty():

        print(
            f"Deleting {deletes.count()} records "
            f"from {silver_table}"
        )

        (
            target.alias("t")
            .merge(
                deletes.alias("d"),
                f"t.{key_column} = d.{key_column}"
            )
            .whenMatchedDelete()
            .execute()
        )

    # ========================================================
    # STEP 2 - INSERT + UPDATE
    # ========================================================

    upserts = (
        changes
        .filter(
            F.col("_change_type").isin(
                "insert",
                "update_postimage"
            )
        )
        .select(*columns)
    )

    if not upserts.isEmpty():

        print(
            f"Upserting {upserts.count()} records "
            f"into {silver_table}"
        )

        # ----------------------------------------------------
        # Build MERGE mappings dynamically
        # ----------------------------------------------------

        update_mapping = {
            column: f"s.{column}"
            for column in columns
            if column != key_column
        }

        insert_mapping = {
            column: f"s.{column}"
            for column in columns
        }

        # ----------------------------------------------------
        # MERGE
        # ----------------------------------------------------

        (
            target.alias("t")
            .merge(
                upserts.alias("s"),
                f"t.{key_column} = s.{key_column}"
            )
            .whenMatchedUpdate(
                set=update_mapping
            )
            .whenNotMatchedInsert(
                values=insert_mapping
            )
            .execute()
        )

    print(
        f"Completed batch {batch_id} "
        f"for {table_name}"
    )

In [0]:
# ============================================================
# GET CURRENT BRONZE VERSION
# ============================================================

def get_current_delta_version(table_name):

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    latest_version = (
        history_df
        .select("version")
        .orderBy(F.col("version").desc())
        .limit(1)
        .collect()[0]["version"]
    )

    return int(latest_version)

In [0]:
# ============================================================
# INITIAL LOAD
# ============================================================

def initial_load(table_name):

    config = TABLE_CONFIG[table_name]

    bronze_table = config["bronze"]
    silver_table = config["silver"]
    columns = config["columns"]

    print("=" * 80)
    print(f"INITIAL LOAD : {table_name}")
    print("=" * 80)

    # --------------------------------------------------------
    # Capture Bronze version BEFORE loading
    # --------------------------------------------------------

    snapshot_version = get_current_delta_version(
        bronze_table
    )

    print(
        f"Bronze snapshot version for "
        f"{table_name}: {snapshot_version}"
    )

    # --------------------------------------------------------
    # Read EXACT Bronze snapshot
    # --------------------------------------------------------

    snapshot_df = (
        spark.read
        .format("delta")
        .option("versionAsOf", snapshot_version)
        .table(bronze_table)
    )

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------

    snapshot_df = clean_string_columns(
        snapshot_df,
        columns
    )

    # --------------------------------------------------------
    # Remove invalid primary keys
    # --------------------------------------------------------

    snapshot_df = snapshot_df.filter(
        F.col(config["key"]).isNotNull()
    )

    # --------------------------------------------------------
    # Deduplicate initial snapshot
    # --------------------------------------------------------

    snapshot_df = snapshot_df.dropDuplicates(
        [config["key"]]
    )

    # --------------------------------------------------------
    # Write to Silver
    # --------------------------------------------------------

    (
        snapshot_df
        .select(*columns)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(silver_table)
    )

    print(
        f"Initial load completed for {table_name}"
    )

    return snapshot_version

In [0]:
# ============================================================
# CHECK WHETHER SILVER TABLE IS EMPTY
# ============================================================

def is_table_empty(table_name):

    return (
        spark.table(table_name)
        .limit(1)
        .count() == 0
    )

In [0]:
# ============================================================
# INITIAL LOAD
#
# IMPORTANT:
# ADF Native CDC should be STOPPED before this section.
# ============================================================

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

snapshot_versions = {}

for table_name, config in TABLE_CONFIG.items():

    silver_table = config["silver"]

    if is_table_empty(silver_table):

        print(
            f"{silver_table} is empty. "
            f"Performing initial load..."
        )

        snapshot_version = initial_load(table_name)

        snapshot_versions[table_name] = snapshot_version

    else:

        print(
            f"{silver_table} already contains data. "
            f"Skipping initial load."
        )

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-4878735151328039>, line 18
     14 for table_name, config in TABLE_CONFIG.items():
     16     silver_table = config["silver"]
---> 18     if is_table_empty(silver_table):
     20         print(
     21             f"{silver_table} is empty. "
     22             f"Performing initial load..."
     23         )
     25         snapshot_version = initial_load(table_name)

File <command-4878735151328037>, line 10, in is_table_empty(table_name)
      5 def is_table_empty(table_name):
      7     return (
      8         spark.table(table_name)
      9         .limit(1)
---> 10         .count() == 0
     11     )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("

In [0]:
# ============================================================
# START CONTINUOUS BRONZE CDF STREAMS
# ============================================================

streams = {}

for table_name, config in TABLE_CONFIG.items():

    bronze_table = config["bronze"]

    checkpoint_path = (
        CHECKPOINT_BASE + table_name
    )

    # --------------------------------------------------------
    # Determine starting version
    # --------------------------------------------------------

    if table_name in snapshot_versions:

        starting_version = (
            snapshot_versions[table_name] + 1
        )

        print(
            f"{table_name}: "
            f"Starting CDF from version "
            f"{starting_version}"
        )

    else:

        # Existing streaming checkpoint will determine
        # the position after the stream has already started.

        starting_version = None

        print(
            f"{table_name}: "
            f"Using existing streaming checkpoint"
        )

    # --------------------------------------------------------
    # Create Bronze CDF stream
    # --------------------------------------------------------

    reader = (
        spark.readStream
        .option("readChangeFeed", "true")
    )

    # Only specify startingVersion for a NEW stream.
    if starting_version is not None:

        reader = reader.option(
            "startingVersion",
            starting_version
        )

    cdf_stream = reader.table(
        bronze_table
    )

    # --------------------------------------------------------
    # Start stream
    # --------------------------------------------------------

    query = (
        cdf_stream
        .writeStream
        .foreachBatch(
            lambda batch_df, batch_id, tn=table_name:
                process_batch(
                    batch_df,
                    batch_id,
                    tn
                )
        )
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .trigger(
            processingTime="30 seconds"
        )
        .start()
    )

    streams[table_name] = query

    print(
        f"Streaming started for {table_name}"
    )

In [0]:
# ============================================================
# WAIT FOR STREAMS
# ============================================================

for table_name, query in streams.items():

    print(
        f"{table_name} stream active: "
        f"{query.id}"
    )

spark.streams.awaitAnyTermination()